# SMLE Estimation for Consumer Revisit Behavior

Trivago RecSys 2019 데이터를 활용한 소비자 재방문 행태 분석을 위한 Simulated Maximum Likelihood Estimation (SMLE) 구현입니다.

## 연구 목표

**핵심 가설**: "가격이 고정되어 있더라도, 소비자의 내부 유틸리티 쇼크가 AR(1) 과정을 따라 진화하며 발생하는 불확실성 때문에 재방문이 발생한다"

## 모델 사양

### 1. Utility Function
$$u_{ijt} = \beta X_j - \alpha p_{ijt} + \epsilon_{ijt}$$

### 2. AR(1) Process
$$\epsilon_{ijt} = \rho \epsilon_{ij,t-\Delta t} + \eta_{ijt}, \quad \eta_{ijt} \sim N(0, \sigma_\eta^2)$$

### 3. Reservation Utility (Closed-form)
$$z_{ijt} = \text{Expected Utility} + \text{Search Option Value}$$

where $m(k) = \phi(k) - k(1-\Phi(k))$ is the search benefit function.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# Try to import numba for speedup (optional)
try:
    from numba import jit
    USE_NUMBA = True
    print("Numba available - using JIT compilation for speedup")
except ImportError:
    USE_NUMBA = False
    # Dummy decorator if numba not available
    def jit(*args, **kwargs):
        def decorator(func):
            return func
        return decorator
    print("Numba not available - running without JIT compilation")

print("Libraries imported successfully!")

## Step 1: Search Benefit Function m(k)

$$m(k) = \phi(k) - k(1 - \Phi(k))$$

where $\phi(k)$ is the PDF and $\Phi(k)$ is the CDF of the standard normal distribution.

In [ ]:
def m_function(k):
    """
    Search Benefit Function: m(k) = φ(k) - k(1 - Φ(k))
    
    Parameters:
    -----------
    k : array-like
        Standardized value (typically (z - μ)/σ)
    
    Returns:
    --------
    m : array-like
        Search benefit value
    """
    k = np.asarray(k)
    phi_k = norm.pdf(k)  # PDF: φ(k)
    Phi_k = norm.cdf(k)  # CDF: Φ(k)
    m = phi_k - k * (1 - Phi_k)
    return m

# Test the function
k_test = np.linspace(-3, 3, 100)
m_test = m_function(k_test)
print(f"m_function tested successfully. Sample values: m(-1)={m_function(-1):.4f}, m(0)={m_function(0):.4f}, m(1)={m_function(1):.4f}")

## Step 2: Simulation Loop for Latent State Evolution

AR(1) 쇼크 경로를 시뮬레이션하여 관찰 불가능한 과거 쇼크를 생성합니다.

In [ ]:
def simulate_shock_path(epsilon_init, rho, sigma_eta, delta_t_values, n_sims):
    """
    Simulate AR(1) shock evolution path for a user-item pair
    
    Parameters:
    -----------
    epsilon_init : float
        Initial shock value
    rho : float
        AR(1) persistence parameter
    sigma_eta : float
        Standard deviation of innovation term
    delta_t_values : array
        Time intervals between actions
    n_sims : int
        Number of simulation draws
    
    Returns:
    --------
    epsilon_paths : array (n_sims, len(delta_t_values))
        Simulated shock paths
    """
    n_periods = len(delta_t_values)
    epsilon_paths = np.zeros((n_sims, n_periods))
    
    for s in range(n_sims):
        epsilon = epsilon_init
        for t_idx, delta_t in enumerate(delta_t_values):
            # AR(1) evolution: ε_{t} = ρ * ε_{t-Δt} + η_t
            # For continuous time, we use: ε_t = ρ^Δt * ε_{t-Δt} + innovation
            # Innovation variance scales with time interval
            rho_scaled = rho ** delta_t  # Scale persistence by time interval
            innovation_std = sigma_eta * np.sqrt(delta_t)
            eta = np.random.normal(0, innovation_std)
            epsilon = rho_scaled * epsilon + eta
            epsilon_paths[s, t_idx] = epsilon
    
    return epsilon_paths

print("simulate_shock_path function defined")

In [ ]:
def compute_reservation_utility_closed_form(expected_utility, sigma_epsilon, c):
    """
    Compute reservation utility z using closed-form solution
    
    z = Expected Utility + Search Option Value
    
    The search option value uses the m(k) function where k = (z - μ)/σ
    
    Parameters:
    -----------
    expected_utility : float
        Expected utility (deterministic part)
    sigma_epsilon : float
        Standard deviation of utility shock
    c : float
        Search cost parameter
    
    Returns:
    --------
    z : float
        Reservation utility
    """
    if sigma_epsilon <= 0:
        return expected_utility
    
    # Solve for z using fixed-point iteration
    # z = μ + σ * m((z - μ)/σ) * (σ/c)
    z = expected_utility  # Initial guess
    tolerance = 1e-6
    max_iter = 100
    
    for _ in range(max_iter):
        k = (z - expected_utility) / sigma_epsilon
        m_k = m_function(k)
        z_new = expected_utility + sigma_epsilon * m_k * (sigma_epsilon / c)
        
        if abs(z_new - z) < tolerance:
            break
        z = z_new
    
    return z

print("compute_reservation_utility_closed_form function defined")

## Step 3: Log-Likelihood Function with Kernel Smoothing

Kernel-smoothed logistic function을 사용하여 smooth한 likelihood를 계산합니다.

In [ ]:
def kernel_smoothed_logistic(x, bandwidth=0.1):
    """
    Kernel-smoothed logistic function for smooth likelihood approximation
    
    Parameters:
    -----------
    x : array-like
        Input values (typically utility differences)
    bandwidth : float
        Smoothing bandwidth parameter
    
    Returns:
    --------
    prob : array-like
        Smoothed probabilities
    """
    x = np.asarray(x)
    # Use logistic CDF with bandwidth scaling
    prob = 1 / (1 + np.exp(-x / bandwidth))
    return prob


def compute_choice_probability(u_best, z_simulated, bandwidth=0.1):
    """
    Compute probability that z > u_best (revisit occurs)
    
    Parameters:
    -----------
    u_best : float
        Best available utility
    z_simulated : array
        Simulated reservation utilities
    bandwidth : float
        Smoothing bandwidth
    
    Returns:
    --------
    prob : float
        Probability of revisit
    """
    # P(revisit) = P(z > u_best) = P(z - u_best > 0)
    diff = z_simulated - u_best
    prob = np.mean(kernel_smoothed_logistic(diff, bandwidth))
    return prob

print("Kernel smoothing functions defined")

In [ ]:
def log_likelihood_user_item(user_item_data, params, prop_cols, n_sims=50, bandwidth=0.1, revisit_col=None):
    """
    Compute log-likelihood for a single user-item pair
    
    Parameters:
    -----------
    user_item_data : DataFrame
        Data for a single user-item pair, sorted by timestamp
    params : dict
        Parameters: {'rho': float, 'c': float, 'alpha': float, 'beta': array}
    prop_cols : list
        List of property column names (prop_...)
    n_sims : int
        Number of simulation draws
    bandwidth : float
        Smoothing bandwidth
    revisit_col : str, optional
        Column name indicating revisit (1 if revisit occurred, 0 otherwise)
        If None, assumes all observations are revisits
    
    Returns:
    --------
    ll : float
        Log-likelihood value
    """
    rho = params['rho']
    c = params['c']
    alpha = params['alpha']
    beta = params['beta']
    
    # Extract data
    prices = user_item_data['price'].values
    delta_t = user_item_data['delta_t'].values
    X = user_item_data[prop_cols].values
    
    # Check if revisit column exists
    if revisit_col and revisit_col in user_item_data.columns:
        revisit_observed = user_item_data[revisit_col].values
    else:
        # If no revisit column, assume all are revisits (delta_t > 0 implies revisit)
        revisit_observed = (delta_t > 0).astype(int)
    
    # Deterministic utility: βX - αp
    deterministic_utility = X @ beta - alpha * prices
    
    # Initialize: first observation has no prior shock
    epsilon_init = 0.0
    sigma_eta = 1.0  # Normalize to 1, or estimate separately
    
    # Simulate shock paths
    epsilon_paths = simulate_shock_path(
        epsilon_init, rho, sigma_eta, delta_t, n_sims
    )
    
    # Compute utilities and reservation utilities for each simulation
    log_probs = []
    
    for t in range(len(user_item_data)):
        # Current deterministic utility
        u_det = deterministic_utility[t]
        
        # For each simulation draw, compute reservation utility
        z_simulated = []
        
        for s in range(n_sims):
            epsilon_t = epsilon_paths[s, t]
            u_t = u_det + epsilon_t
            
            # Compute reservation utility
            # Expected utility = deterministic part (since E[ε] = 0)
            expected_u = u_det
            sigma_eps = sigma_eta  # Standard deviation of current shock
            
            z_t = compute_reservation_utility_closed_form(
                expected_u, sigma_eps, c
            )
            z_simulated.append(z_t)
        
        z_simulated = np.array(z_simulated)
        
        # Best available utility (current utility with shock)
        u_best = u_det + np.mean(epsilon_paths[:, t])
        
        # Probability of revisit: P(z > u_best)
        prob_revisit = compute_choice_probability(u_best, z_simulated, bandwidth)
        
        # Avoid log(0)
        prob_revisit = np.clip(prob_revisit, 1e-10, 1 - 1e-10)
        
        # Use observed revisit status
        if revisit_observed[t] == 1:
            # Revisit occurred: P(z > u_best)
            log_prob = np.log(prob_revisit)
        else:
            # No revisit: P(z <= u_best) = 1 - P(z > u_best)
            log_prob = np.log(1 - prob_revisit)
        
        log_probs.append(log_prob)
    
    return np.sum(log_probs)


def log_likelihood_full(sample_df, params, prop_cols, n_sims=50, bandwidth=0.1, revisit_col=None):
    """
    Compute full log-likelihood across all user-item pairs
    
    Parameters:
    -----------
    sample_df : DataFrame
        Full dataset with columns: user_id, reference, delta_t, price, prop_..., timestamp
    params : dict
        Parameters: {'rho': float, 'c': float, 'alpha': float, 'beta': array}
    prop_cols : list
        List of property column names
    n_sims : int
        Number of simulation draws
    bandwidth : float
        Smoothing bandwidth
    revisit_col : str, optional
        Column name indicating revisit (1 if revisit occurred, 0 otherwise)
    
    Returns:
    --------
    ll : float
        Total log-likelihood
    """
    total_ll = 0.0
    
    # Group by user_id and reference (item)
    grouped = sample_df.groupby(['user_id', 'reference'])
    
    for (user_id, reference), group_data in grouped:
        # Sort by timestamp
        group_data = group_data.sort_values('timestamp').reset_index(drop=True)
        
        try:
            ll_user_item = log_likelihood_user_item(
                group_data, params, prop_cols, n_sims, bandwidth, revisit_col
            )
            total_ll += ll_user_item
        except Exception as e:
            # Skip problematic cases
            print(f"Warning: Error for user {user_id}, item {reference}: {e}")
            continue
    
    return total_ll

print("Log-likelihood functions defined")

## Step 4: Optimization

scipy.optimize.minimize를 사용하여 파라미터를 추정합니다.

In [ ]:
def objective_function(params_vector, sample_df, prop_cols, n_sims=50, bandwidth=0.1, revisit_col=None):
    """
    Objective function for optimization (negative log-likelihood)
    
    Parameters:
    -----------
    params_vector : array
        [rho, c, alpha, beta_1, beta_2, ..., beta_n]
    sample_df : DataFrame
        Full dataset
    prop_cols : list
        List of property column names
    n_sims : int
        Number of simulation draws
    bandwidth : float
        Smoothing bandwidth
    revisit_col : str, optional
        Column name indicating revisit
    
    Returns:
    --------
    neg_ll : float
        Negative log-likelihood (to minimize)
    """
    # Unpack parameters
    rho = params_vector[0]
    c = params_vector[1]
    alpha = params_vector[2]
    beta = params_vector[3:]
    
    # Ensure constraints
    rho = np.clip(rho, -0.99, 0.99)  # Stationarity constraint
    c = np.maximum(c, 1e-6)  # Positive search cost
    alpha = np.maximum(alpha, 1e-6)  # Positive price sensitivity
    
    params = {
        'rho': rho,
        'c': c,
        'alpha': alpha,
        'beta': beta
    }
    
    # Compute log-likelihood
    ll = log_likelihood_full(sample_df, params, prop_cols, n_sims, bandwidth, revisit_col)
    
    return -ll  # Return negative for minimization

print("Objective function defined")

In [ ]:
def estimate_smle(sample_df, prop_cols, n_sims=50, bandwidth=0.1, 
                  initial_params=None, method='L-BFGS-B', revisit_col=None):
    """
    Estimate parameters using Simulated Maximum Likelihood
    
    Parameters:
    -----------
    sample_df : DataFrame
        Full dataset
    prop_cols : list
        List of property column names (e.g., ['prop_1', 'prop_2', ...])
    n_sims : int
        Number of simulation draws (default: 50)
    bandwidth : float
        Smoothing bandwidth (default: 0.1)
    initial_params : dict, optional
        Initial parameter values. If None, uses defaults:
        {'rho': 0.5, 'c': 1.0, 'alpha': 0.1, 'beta': zeros}
    method : str
        Optimization method (default: 'L-BFGS-B')
    revisit_col : str, optional
        Column name indicating revisit (1 if revisit occurred, 0 otherwise)
        If None, infers from delta_t > 0
    
    Returns:
    --------
    result : OptimizeResult
        Optimization result from scipy.optimize.minimize
    """
    n_props = len(prop_cols)
    
    # Set initial parameters
    if initial_params is None:
        initial_params = {
            'rho': 0.5,
            'c': 1.0,
            'alpha': 0.1,
            'beta': np.zeros(n_props)
        }
    
    # Pack parameters into vector
    params_vector = np.concatenate([
        [initial_params['rho']],
        [initial_params['c']],
        [initial_params['alpha']],
        initial_params['beta']
    ])
    
    # Set bounds
    bounds = [
        (-0.99, 0.99),  # rho: stationarity
        (1e-6, None),    # c: positive
        (1e-6, None),    # alpha: positive
    ] + [(-10, 10)] * n_props  # beta: reasonable range
    
    # Optimize
    print("Starting optimization...")
    print(f"Initial parameters: rho={initial_params['rho']:.3f}, "
          f"c={initial_params['c']:.3f}, alpha={initial_params['alpha']:.3f}")
    print(f"Number of simulations: {n_sims}")
    print(f"Bandwidth: {bandwidth}")
    
    result = minimize(
        objective_function,
        params_vector,
        args=(sample_df, prop_cols, n_sims, bandwidth, revisit_col),
        method=method,
        bounds=bounds,
        options={'maxiter': 100, 'disp': True}
    )
    
    # Unpack results
    rho_est = result.x[0]
    c_est = result.x[1]
    alpha_est = result.x[2]
    beta_est = result.x[3:]
    
    print("\n" + "="*50)
    print("Estimation Results:")
    print("="*50)
    print(f"rho (AR persistence): {rho_est:.4f}")
    print(f"c (search cost): {c_est:.4f}")
    print(f"alpha (price sensitivity): {alpha_est:.4f}")
    print(f"beta (property coefficients):")
    for i, prop in enumerate(prop_cols):
        print(f"  {prop}: {beta_est[i]:.4f}")
    print(f"\nLog-likelihood: {-result.fun:.4f}")
    print(f"Success: {result.success}")
    print("="*50)
    
    return result

print("estimate_smle function defined")

## 사용 예제

아래 코드는 실제 데이터를 사용하여 파라미터를 추정하는 예제입니다.

**데이터 요구사항:**
- `user_id`: 사용자 식별자
- `reference`: 아이템 식별자
- `delta_t`: 동일 유저가 동일 아이템을 다시 클릭하기까지 걸린 시간 (hours)
- `price`: 아이템 가격
- `prop_1`, `prop_2`, ..., `prop_N`: 아이템의 특성
- `timestamp`: 액션 발생 시점
- `revisit` (optional): 재방문 여부 (1: 재방문, 0: 미재방문)

In [ ]:
# ============================================================================
# 데이터 로드 및 준비
# ============================================================================

# 예제: sample_df가 이미 메모리에 로드되어 있다고 가정
# 실제 사용 시에는 아래와 같이 데이터를 로드하세요:
# sample_df = pd.read_csv('your_data.csv')

# 속성 컬럼 식별
# prop_cols = [col for col in sample_df.columns if col.startswith('prop_')]

# 예제 (실제 데이터가 있을 때 주석 해제):
"""
# 1. 데이터 로드
sample_df = pd.read_csv('trivago_recsys_2019_processed.csv')

# 2. 속성 컬럼 식별
prop_cols = [col for col in sample_df.columns if col.startswith('prop_')]
print(f"Found {len(prop_cols)} property columns: {prop_cols[:5]}...")

# 3. 데이터 확인
print(f"\nData shape: {sample_df.shape}")
print(f"\nColumns: {sample_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(sample_df.head())
"""

In [ ]:
# ============================================================================
# 파라미터 추정 실행
# ============================================================================

# 실제 사용 시 아래 코드의 주석을 해제하세요:
"""
# 파라미터 추정
result = estimate_smle(
    sample_df=sample_df,
    prop_cols=prop_cols,
    n_sims=50,           # 시뮬레이션 횟수 (정확도 향상을 위해 증가 가능)
    bandwidth=0.1,       # Smoothing bandwidth
    revisit_col='revisit',  # 재방문 컬럼 (없으면 delta_t > 0으로 추론)
    initial_params={     # 초기값 (optional)
        'rho': 0.5,
        'c': 1.0,
        'alpha': 0.1,
        'beta': np.zeros(len(prop_cols))
    }
)

# 결과 추출
rho_est = result.x[0]      # AR(1) 지속성
c_est = result.x[1]        # 검색 비용
alpha_est = result.x[2]    # 가격 민감도
beta_est = result.x[3:]    # 속성 계수들

print(f"\n\n최종 추정 결과:")
print(f"rho (AR persistence): {rho_est:.4f}")
print(f"c (search cost): {c_est:.4f}")
print(f"alpha (price sensitivity): {alpha_est:.4f}")
print(f"\nLog-likelihood: {-result.fun:.4f}")
"""

## 파라미터 설명

### 추정 파라미터

- **`rho`**: AR(1) 지속성 파라미터
  - 범위: (-0.99, 0.99) - stationarity 제약
  - 해석: 높을수록 과거 쇼크가 현재에 더 큰 영향을 미침

- **`c`**: 검색 비용 (Search Cost)
  - 범위: (0, ∞)
  - 해석: 높을수록 검색 비용이 높아 재방문 확률 감소

- **`alpha`**: 가격 민감도
  - 범위: (0, ∞)
  - 해석: 높을수록 가격에 더 민감하게 반응

- **`beta`**: 속성 계수 벡터
  - 각 `prop_*` 컬럼에 대한 계수
  - 해석: 각 속성이 유틸리티에 미치는 영향

### 하이퍼파라미터

- **`n_sims`**: 시뮬레이션 횟수 (기본값: 50)
  - 증가시키면 정확도 향상, 하지만 계산 시간 증가
  - 권장값: 50-200

- **`bandwidth`**: Smoothing bandwidth (기본값: 0.1)
  - Kernel-smoothed logistic function의 smoothing 파라미터
  - 작을수록 더 날카로운 확률 분포

## 주의사항

1. **계산 시간**: 시뮬레이션 기반 추정이므로 데이터 크기에 따라 시간이 오래 걸릴 수 있습니다.
   - `n_sims`를 줄이면 속도 향상, 하지만 정확도 감소
   - 병렬화를 고려할 수 있습니다 (향후 개선 가능)

2. **수렴성**: 초기값에 따라 최적화가 수렴하지 않을 수 있습니다.
   - 여러 초기값으로 시도해보세요
   - `method='Nelder-Mead'`를 사용하면 더 robust할 수 있습니다

3. **데이터 품질**: 
   - `delta_t`가 0인 경우는 첫 방문으로 간주됩니다
   - 각 user-item 쌍에 최소 2개 이상의 관측이 필요합니다